# Comprehensive Guide to Inferential Statistics in Python

This notebook explores the core inferential statistics used in data science. Inferential statistics allow us to make predictions or inferences about a population based on a sample of data.

**Covered in this Notebook:**
1. Setup and Understanding Our Mock Data
2. The Parametric vs. Non-Parametric Reference Guide
3. T-Tests (One-Sample, Independent, Paired)
4. Analysis of Variance (ANOVA)
5. Non-Parametric Tests (Mann-Whitney U, Chi-Square)
6. Correlation and Regression

---

## 1. Setup and Understanding Our Mock Data

Before running any statistical tests, we must understand our data. In the real world, you would perform Exploratory Data Analysis (EDA) first. Here, we are generating mock data using `numpy`. We use a random seed (`np.random.seed(42)`) to ensure that every time you run this notebook, the "random" numbers generated are exactly the same, making our results reproducible.

**Our Dataset Contains the Following Variables:**

* **`group_a` and `group_b` (Continuous):** Generated using a normal (Gaussian) distribution. These represent two independent groups (e.g., test scores from two different classrooms). We will use these for independent T-tests and ANOVA.
* **`before_treatment` and `after_treatment` (Continuous, Paired):** These represent measurements taken from the *same* subjects at two different times (e.g., blood pressure before and after medication). This dependence requires specific paired tests.
* **`category_1` and `category_2` (Categorical):** Randomly assigned text labels (e.g., color preferences and pass/fail status). We use these to test for associations between categories using tests like the Chi-Square.
* **`years_experience` (Continuous) and `salary` (Continuous, Correlated):** We generate years of experience uniformly, and then mathematically force `salary` to increase as experience increases. We do this specifically so our correlation and regression tests later on have a clear linear relationship to discover.

In [11]:
# Import necessary libraries
import numpy as np
import pandas as pd
import scipy.stats as stats
import statsmodels.api as sm
from statsmodels.formula.api import ols
import matplotlib.pyplot as plt
import seaborn as sns

# Set visual style
sns.set_theme(style="whitegrid")

# Set random seed for reproducibility
np.random.seed(42)

# Generate Mock Data
data = pd.DataFrame({
    'group_a': np.random.normal(loc=50, scale=10, size=100),
    'group_b': np.random.normal(loc=55, scale=10, size=100),
    'before_treatment': np.random.normal(loc=60, scale=8, size=100),
    'after_treatment': np.random.normal(loc=65, scale=8, size=100), # Paired with before
    'category_1': np.random.choice(['Red', 'Blue', 'Green'], size=100),
    'category_2': np.random.choice(['Pass', 'Fail'], size=100),
    'salary': np.random.normal(loc=60000, scale=15000, size=100),
    'years_experience': np.random.uniform(1, 20, size=100)
})

# Add a linear relationship for regression
data['salary'] = data['salary'] + (data['years_experience'] * 2500)

print("Data successfully generated. Ready for testing!")

Data successfully generated. Ready for testing!


In [12]:
display(data.head())
display(data.tail())

,group_a,group_b,before_treatment,after_treatment,category_1,category_2,salary,years_experience
0,54.967142,40.846293,62.862299,58.368040,Green,Pass,86413.463888,15.792867
1,48.617357,50.793547,64.486276,60.518552,Green,Pass,65897.337904,3.719432
2,56.476885,51.572855,68.664410,70.978349,Red,Pass,60649.570068,4.886347
3,65.230299,46.977227,68.430416,69.882962,Red,Pass,105060.662898,14.567218
4,47.658466,53.387143,48.978645,64.832787,Green,Fail,103808.117358,10.385648


,group_a,group_b,before_treatment,after_treatment,category_1,category_2,salary,years_experience
95,35.364851,58.853174,54.456723,61.246595,Blue,Pass,118592.272307,16.894731
96,52.961203,46.161426,67.196799,51.294924,Green,Fail,115991.419091,12.161618
97,52.610553,56.537251,62.458396,75.830979,Green,Fail,75982.168499,6.586849
98,50.051135,55.582087,66.502897,64.083681,Green,Fail,108840.114346,14.567125
99,47.654129,43.570297,65.037031,74.902530,Green,Pass,102711.691598,11.025307


## 1. T-Tests
T-tests are used to compare the means of groups.

* **One-Sample T-Test:** Tests if the mean of a single group is equal to a known value.
* **Independent Two-Sample T-Test:** Tests if the means of two independent groups are significantly different.
* **Paired T-Test:** Tests if the means of two related groups (e.g., same subjects before/after) are significantly different.

In [13]:
# --- 1A. One-Sample T-Test ---
# Hypothesis: Is the mean of 'group_a' significantly different from 48?
t_stat_1, p_val_1 = stats.ttest_1samp(data['group_a'], popmean=48)
print(f"One-Sample T-Test: t-statistic = {t_stat_1:.3f}, p-value = {p_val_1:.4f}")

# --- 1B. Independent Two-Sample T-Test ---
# Hypothesis: Are the means of 'group_a' and 'group_b' different?
t_stat_ind, p_val_ind = stats.ttest_ind(data['group_a'], data['group_b'], equal_var=False) # Welch's t-test
print(f"Independent T-Test: t-statistic = {t_stat_ind:.3f}, p-value = {p_val_ind:.4f}")

# --- 1C. Paired T-Test ---
# Hypothesis: Did the treatment have an effect? (comparing before vs after)
t_stat_pair, p_val_pair = stats.ttest_rel(data['before_treatment'], data['after_treatment'])
print(f"Paired T-Test: t-statistic = {t_stat_pair:.3f}, p-value = {p_val_pair:.4f}")

One-Sample T-Test: t-statistic = 1.059, p-value = 0.2923
Independent T-Test: t-statistic = -4.755, p-value = 0.0000
Paired T-Test: t-statistic = -4.767, p-value = 0.0000


## 2. Analysis of Variance (ANOVA)
ANOVA is used to compare the means of three or more independent groups.

* **One-Way ANOVA:** Tests if there are statistically significant differences between the means of three or more independent groups based on one factor.

In [14]:
# Create a 3rd group for ANOVA purposes
group_c = np.random.normal(loc=48, scale=10, size=100)

# Hypothesis: Are the means of group_a, group_b, and group_c equal?
f_stat, p_val_anova = stats.f_oneway(data['group_a'], data['group_b'], group_c)
print(f"One-Way ANOVA: F-statistic = {f_stat:.3f}, p-value = {p_val_anova:.4f}")

One-Way ANOVA: F-statistic = 12.536, p-value = 0.0000


## 3. Non-Parametric Tests
Used when the data does not meet the assumptions of parametric tests (e.g., not normally distributed, ordinal data, or categorical data).

* **Mann-Whitney U Test:** The non-parametric equivalent of the independent t-test.
* **Chi-Square Test of Independence:** Tests if there is a significant association between two categorical variables.

In [15]:
# --- 3A. Mann-Whitney U Test ---
# Hypothesis: Is the distribution of group_a equal to group_b?
u_stat, p_val_mw = stats.mannwhitneyu(data['group_a'], data['group_b'])
print(f"Mann-Whitney U: U-statistic = {u_stat:.3f}, p-value = {p_val_mw:.4f}")

# --- 3B. Chi-Square Test of Independence ---
# Create a contingency table first
contingency_table = pd.crosstab(data['category_1'], data['category_2'])

# Hypothesis: Are category_1 and category_2 independent?
chi2_stat, p_val_chi, dof, expected = stats.chi2_contingency(contingency_table)
print(f"Chi-Square Test: Chi2-statistic = {chi2_stat:.3f}, p-value = {p_val_chi:.4f}")

Mann-Whitney U: U-statistic = 3300.000, p-value = 0.0000
Chi-Square Test: Chi2-statistic = 0.348, p-value = 0.8405


## 4. Correlation and Regression
Used to evaluate relationships between numerical variables.

* **Pearson Correlation:** Measures linear relationship between two continuous variables.
* **Spearman Rank Correlation:** Non-parametric measure of rank correlation (monotonic relationships).
* **Ordinary Least Squares (OLS) Regression:** Models the linear relationship between a dependent variable and one or more independent variables.

In [16]:
# --- 4A. Correlation ---
# Pearson (Linear)
corr_p, p_val_p = stats.pearsonr(data['years_experience'], data['salary'])
print(f"Pearson Correlation: r = {corr_p:.3f}, p-value = {p_val_p:.4f}")

# Spearman (Non-linear/Rank based)
corr_s, p_val_s = stats.spearmanr(data['years_experience'], data['salary'])
print(f"Spearman Correlation: rho = {corr_s:.3f}, p-value = {p_val_s:.4f}")

# --- 4B. Simple Linear Regression (Statsmodels) ---
# Predict salary based on years of experience
# Add a constant for the intercept
X = sm.add_constant(data['years_experience']) 
y = data['salary']

model = sm.OLS(y, X).fit()
print("\n--- OLS Regression Summary ---")
print(model.summary())

Pearson Correlation: r = 0.646, p-value = 0.0000
Spearman Correlation: rho = 0.641, p-value = 0.0000

--- OLS Regression Summary ---
                            OLS Regression Results                            
Dep. Variable:                 salary   R-squared:                       0.417
Model:                            OLS   Adj. R-squared:                  0.411
Method:                 Least Squares   F-statistic:                     70.01
Date:                Mon, 23 Feb 2026   Prob (F-statistic):           4.15e-13
Time:                        09:43:42   Log-Likelihood:                -1110.6
No. Observations:                 100   AIC:                             2225.
Df Residuals:                      98   BIC:                             2230.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                       coef    std err          t      P>|t|      [0.025     

---

## 5. The Parametric vs. Non-Parametric Reference Guide

When choosing a statistical test, you must first decide between a parametric and a non-parametric method. 

**Parametric Methods** assume that your data follows a specific distribution (usually a normal distribution). They are more powerful but have strict assumptions (e.g., continuous data, equal variances, no extreme outliers).

**Non-Parametric Methods** are "distribution-free." They do not assume a normal distribution. You use these when your data is skewed, has heavy outliers, or consists of ordinal (ranked) data.

Here is a comprehensive mapping of common statistical goals to their appropriate tests:

| Statistical Goal | Parametric Method | Non-Parametric Equivalent |
| :--- | :--- | :--- |
| **Compare 1 group to a known value** | One-Sample T-Test | Wilcoxon Signed-Rank Test |
| **Compare 2 independent groups** | Independent Two-Sample T-Test | Mann-Whitney U Test |
| **Compare 2 related/paired groups** | Paired T-Test | Wilcoxon Signed-Rank Test |
| **Compare 3+ independent groups** | One-Way ANOVA | Kruskal-Wallis H Test |
| **Compare 3+ related groups** | Repeated Measures ANOVA | Friedman Test |
| **Measure relationship between 2 variables** | Pearson Correlation | Spearman Rank Correlation |
| **Predict a variable based on others** | Simple/Multiple Linear Regression | Non-parametric Regression / Decision Trees |
| **Test association of categorical variables** | N/A (Requires continuous data) | Chi-Square Test of Independence |

---
### 5A. Comparing One Group to a Known Value
**Goal:** Determine if a single sample's central tendency differs significantly from a known population value (e.g., testing if a factory's average part size equals 50mm).

In [17]:
known_value = 48

# Parametric: One-Sample T-Test (tests the mean)
t_stat_1, p_val_t1 = stats.ttest_1samp(data['group_a'], popmean=known_value)
print(f"One-Sample T-Test: t-statistic = {t_stat_1:.3f}, p-value = {p_val_t1:.4f}")

# Non-Parametric: Wilcoxon Signed-Rank Test (tests the median against a known value)
# We subtract the known value from the data to test if the median difference is 0
w_stat_1, p_val_w1 = stats.wilcoxon(data['group_a'] - known_value)
print(f"Wilcoxon 1-Sample Test: w-statistic = {w_stat_1:.3f}, p-value = {p_val_w1:.4f}")

One-Sample T-Test: t-statistic = 1.059, p-value = 0.2923
Wilcoxon 1-Sample Test: w-statistic = 2178.000, p-value = 0.2328


---
### 5B. Comparing Two Independent Groups
**Goal:** Determine if two entirely separate groups (e.g., Control Group vs. Test Group) have significantly different central tendencies.